# DAY 6 — Advanced Analytics + Risk Metrics
## Capstone Project - I | Bluestock Mutual Fund Analytics

| Task | Description | Output |
|---|---|---|
| 1 | Historical VaR (95%) + CVaR for all 40 schemes | `var_cvar_report.csv` |
| 2 | Rolling 90-day Sharpe — top 5 funds over time | `rolling_sharpe_chart.png` |
| 3 | Investor cohort analysis (first-txn-year grouping) | Console + DataFrame |
| 4 | SIP continuity analysis — at-risk investor flagging | `sip_continuity.csv` |
| 5 | Simple fund recommender (Low / Moderate / High) | `recommender.py` |
| 6 | Sector HHI concentration per equity fund | `sector_hhi.csv` |
| 7 | 5 advanced narrative insights | `advanced_insights.md` |

> **All thresholds and paths are configured in `config.py` — nothing is hardcoded.**

In [ ]:
import os, sys
# Ensure project root is on path
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from IPython.display import Image, Markdown, display

import config as C
import advanced_analytics as aa
from recommender import recommend_funds, recommend_all

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
print(f'Project root : {C.PROJECT_ROOT}')
print(f'VaR confidence: {C.VAR_CONFIDENCE*100:.0f}%')
print(f'Rolling window: {C.ROLLING_WINDOW} days')
print(f'SIP gap threshold: {C.SIP_GAP_THRESHOLD} days')

## Load Shared Data

In [ ]:
returns     = aa.load_nav_returns()
fund_master = aa.load_fund_master()
txn         = aa.load_transactions()
holdings    = aa.load_holdings()
scorecard   = aa.load_scorecard()

print(f'Daily returns : {returns.shape[0]} dates × {returns.shape[1]} funds')
print(f'Investors     : {txn.investor_id.nunique():,}')
print(f'Holdings rows : {len(holdings)}')
returns.describe().T[['mean','std','min','max']].round(4)

## Task 1 — Historical VaR (95%) + CVaR

- **VaR (95%)** = 5th percentile of daily return distribution
- **CVaR** = mean of all returns *below* the VaR threshold (Expected Shortfall)
- Both represent daily loss magnitude; higher absolute value = more risk
- Computed for all 40 schemes using `config.VAR_CONFIDENCE = 0.95`

In [ ]:
var_df = aa.compute_var_cvar(returns, fund_master)

# Display sorted by risk
display_cols = ['scheme_name','category','risk_category','var_95_pct','cvar_95_pct','daily_std_pct']
var_df[display_cols].style \
    .format({'var_95_pct':'{:.3f}%','cvar_95_pct':'{:.3f}%','daily_std_pct':'{:.3f}%'}) \
    .background_gradient(subset=['var_95_pct'], cmap='RdYlGn')

In [ ]:
# Visualise VaR vs CVaR
aa.plot_var_cvar(var_df)
Image(str(C.chart_path('var_cvar_chart.png')))

In [ ]:
# Return distribution for worst and best VaR funds
worst_code = var_df.nsmallest(1,'var_95_pct').iloc[0]['amfi_code']
best_code  = var_df.nlargest(1,'var_95_pct').iloc[0]['amfi_code']

fig, axes = plt.subplots(1,2,figsize=(14,5))
for ax, code, label_suffix in zip(axes, [worst_code, best_code],
                                   ['Highest Risk','Lowest Risk']):
    r   = returns[code].dropna() * 100
    var = float(np.percentile(r, C.VAR_PERCENTILE*100))
    ax.hist(r, bins=60, color='#1565C0', alpha=0.7, edgecolor='white')
    ax.axvline(var, color='#D32F2F', linewidth=2, label=f'VaR={var:.2f}%')
    ax.fill_betweenx([0,ax.get_ylim()[1] if ax.get_ylim()[1]>0 else 500],
                      r.min(), var, alpha=0.2, color='#D32F2F')
    name = fund_master[fund_master.amfi_code==code].iloc[0]['scheme_name'][:30]
    ax.set_title(f'{label_suffix}\n{name}', fontweight='bold')
    ax.set_xlabel('Daily Return (%)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.4)
plt.suptitle('Daily Return Distributions — VaR (95%) Highlighted', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Task 2 — Rolling 90-Day Sharpe Ratio

$$\text{Rolling Sharpe} = \frac{\bar{r}_{90} - R_f}{\sigma_{90}} \times \sqrt{252}$$

Computed for top `ROLLING_SHARPE_FUNDS` funds by scorecard rank.

In [ ]:
rs_df = aa.compute_rolling_sharpe(returns, scorecard, fund_master)
aa.plot_rolling_sharpe(rs_df)
Image(str(C.chart_path('rolling_sharpe_chart.png')))

In [ ]:
print('Rolling Sharpe summary statistics:')
rs_df.describe().round(3)

## Task 3 — Investor Cohort Analysis

Group investors by **year of first transaction**.  
For each cohort: investor count, avg SIP amount, total invested, top fund preference.

In [ ]:
cohort_df = aa.investor_cohort_analysis(txn, fund_master)
cohort_df.style.format({
    'avg_sip_amount': '₹{:,.0f}',
    'total_invested':  '₹{:,.0f}'
})

In [ ]:
# Cohort comparison chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(cohort_df['cohort_year'].astype(str), cohort_df['investor_count'],
            color=['#1565C0','#00ACC1'], width=0.4, edgecolor='white')
axes[0].set_title('Investors per Cohort Year', fontweight='bold')
axes[0].set_ylabel('Investor Count')

axes[1].bar(cohort_df['cohort_year'].astype(str), cohort_df['avg_sip_amount'],
            color=['#F9A825','#00897B'], width=0.4, edgecolor='white')
axes[1].set_title('Avg SIP Amount by Cohort Year', fontweight='bold')
axes[1].set_ylabel('Avg SIP (₹)')
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f'₹{x:,.0f}'))

plt.suptitle('Investor Cohort Analysis', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

## Task 4 — SIP Continuity Analysis

For investors with ≥ `SIP_MIN_TRANSACTIONS` SIP transactions:  
- Compute **average gap** (days) between consecutive SIP instalments  
- Flag **at-risk** if avg gap > `SIP_GAP_THRESHOLD` days

In [ ]:
continuity_df = aa.sip_continuity_analysis(txn)
print(f'At-risk investors: {continuity_df.at_risk.sum()} / {len(continuity_df)}')
continuity_df.head(10)

In [ ]:
# Gap distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(continuity_df['avg_gap_days'], bins=40,
             color='#1565C0', alpha=0.8, edgecolor='white')
axes[0].axvline(C.SIP_GAP_THRESHOLD, color='#D32F2F',
                linewidth=2, linestyle='--', label=f'Threshold ({C.SIP_GAP_THRESHOLD}d)')
axes[0].set_title('Distribution of Avg SIP Gap (Days)', fontweight='bold')
axes[0].set_xlabel('Average Gap (Days)')
axes[0].legend()

risk_counts = continuity_df['at_risk'].value_counts()
axes[1].pie(risk_counts.values,
            labels=['At-Risk' if v else 'Healthy' for v in risk_counts.index],
            colors=['#D32F2F','#00897B'],
            autopct='%1.1f%%', startangle=90,
            wedgeprops=dict(edgecolor='white',linewidth=2))
axes[1].set_title('SIP Continuity Status', fontweight='bold')

plt.suptitle('SIP Continuity Analysis', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

## Task 5 — Simple Fund Recommender

**Input** : Risk appetite (`Low` / `Moderate` / `High`)  
**Logic** : Filter by `risk_category` via `config.RISK_APPETITE_MAP`, rank by Sharpe ratio  
**Output**: Top `RECOMMENDER_TOP_N` funds with full metrics  

Run `python recommender.py --risk Moderate` from the terminal.

In [ ]:
# Low risk
low_recs = recommend_funds('Low', verbose=True)

In [ ]:
# Moderate risk
mod_recs = recommend_funds('Moderate', verbose=True)

In [ ]:
# High risk
high_recs = recommend_funds('High', verbose=True)

## Task 6 — Sector HHI Concentration

$$\text{HHI} = \sum_{i} w_i^2$$

where $w_i$ = weight of sector $i$ in the fund (%).  
Scale: 100–10,000. Higher = more concentrated.  

| Range | Interpretation |
|---|---|
| HHI ≥ 2500 | Highly Concentrated |
| 1500 ≤ HHI < 2500 | Moderately Concentrated |
| HHI < 1500 | Diversified |

In [ ]:
hhi_df = aa.compute_sector_hhi(holdings, fund_master)
aa.plot_hhi(hhi_df)
Image(str(C.chart_path('sector_hhi_chart.png')))

In [ ]:
hhi_df[['scheme_name','hhi','concentration','top_sector','top_sector_wt_pct','n_sectors']] \
    .style.format({'hhi':'{:.0f}','top_sector_wt_pct':'{:.1f}%'}) \
    .background_gradient(subset=['hhi'], cmap='RdYlGn_r')

## Task 7 — 5 Advanced Insights

In [ ]:
insights = aa.generate_advanced_insights(var_df, cohort_df, continuity_df, hhi_df, scorecard)
for ins in insights:
    display(Markdown('> ' + ins))
    display(Markdown('---'))

## Summary — All Deliverables

In [ ]:
deliverables = [
    'reports/var_cvar_report.csv',
    'reports/sip_continuity.csv',
    'reports/sector_hhi.csv',
    'reports/advanced_insights.md',
    'reports/charts/rolling_sharpe_chart.png',
    'reports/charts/var_cvar_chart.png',
    'reports/charts/sector_hhi_chart.png',
    'recommender.py',
    'notebooks/Advanced_Analytics.ipynb',
]
print('Day 6 Deliverables:')
for p in deliverables:
    full = C.PROJECT_ROOT / p
    status = '✅' if full.exists() else '❌'
    size   = full.stat().st_size // 1024 if full.exists() else 0
    print(f'  {status} {p}  ({size} KB)')